# 16. Venn Diagrams & UpSet Plots (Set Overlap Analysis)

This interactive notebook demonstrates multi-set overlap visualizations in Python using **`venny4py`** for 2–4 sets and **`upsetplot`** for 5+ sample comparisons with **simulated random data**.

### Overview
- **Purpose**: Visualize set overlaps, intersections, and unique elements across multiple sample cohorts, differentially expressed gene lists, or detected proteomics features.
- **Use Case**: Comparing candidate biomarkers across 2–4 conditions (Venn diagrams) or scalable multi-group intersection discovery across 5–20+ cohorts (UpSet plots).

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from upsetplot import UpSet, from_contents
from venny4py.venny4py import venny4py

### Helper Functions & Synthetic Set Generation

In [ ]:
def generate_random_sets_4way(seed: int = 42) -> dict[str, set[str]]:
    """Generates 4 simulated biomarker candidate sets with shared core and specific hits."""
    rng = random.Random(seed)
    universe = [f"Protein_{i+1:03d}" for i in range(200)]
    
    core = set(rng.sample(universe, 25))
    available = list(set(universe) - core)
    
    sets = {}
    for name in ["Control_vs_Mild", "Control_vs_Severe", "Treated_Arm_A", "Treated_Arm_B"]:
        extra = set(rng.sample(available, rng.randint(30, 60)))
        sets[name] = core | extra
    return sets


def generate_random_samples_10way(n_samples: int = 10, seed: int = 42) -> dict[str, set[str]]:
    """Generates 10 simulated patient sample feature sets for UpSet plotting."""
    rng = random.Random(seed)
    universe = [f"Gene_{i+1:04d}" for i in range(1000)]
    shared_core = set(rng.sample(universe, 40))
    available = list(set(universe) - shared_core)
    
    samples = {}
    for i in range(1, n_samples + 1):
        extra = set(rng.sample(available, rng.randint(80, 180)))
        samples[f"Sample_{i:02d}"] = shared_core | extra
    return samples

### Part 1: Four-Way Venn Diagram (`venny4py`)

In [ ]:
sets_4way = generate_random_sets_4way()
out_dir_venn = Path("venn_4_output")
out_dir_venn.mkdir(exist_ok=True)

venny4py(sets=sets_4way, out=str(out_dir_venn), ext="png", dpi=300)
print("4-Way Venn Diagram successfully generated in:", out_dir_venn.resolve())

### Part 2: Scalable 10-Sample UpSet Plot (`upsetplot`)

In [ ]:
samples_10way = generate_random_samples_10way()
upset_data = from_contents(samples_10way)

upset = UpSet(
    upset_data,
    subset_size="count",
    show_counts=True,
    sort_by="cardinality",
    sort_categories_by="cardinality",
    min_subset_size=5,
    max_subset_rank=20,
    facecolor="#2C7FB8"
)

fig = plt.figure(figsize=(15, 8))
upset.plot(fig=fig)
plt.show()

core_all = set.intersection(*samples_10way.values())
print(f"Gemeinsamer Core über alle 10 Samples: {len(core_all)} Gene")